# Session 1: Introduction to LangChain as an Orchestrator

This notebook covers the transition from raw API SDKs (Google GenAI / OpenAI) to using LangChain as a unified orchestrator. We will cover:
1. **Direct SDK comparison** (OpenAI vs. Gemini API client).
2. **LangChain Chat Models** & message structures (`SystemMessage`, `HumanMessage`, `AIMessage`).
3. **State Management**: Building a manual conversational memory loop.
4. **Behind the Scenes**: Enabling LangChain debugging and verbose logging.
5. **Model Configurations**: Setting up parameters like temperature and max tokens.

## Setup & Installation

Install the necessary dependencies and configure your environment variables. This setup is compatible with both local `.env` files and Google Colab Secrets.

In [ ]:
# !pip install google-genai openai langchain langchain-google-genai python-dotenv

In [ ]:
import os

# 1. Try to load local .env if it exists
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# 2. Try to load Colab secrets if running in Google Colab
if "GOOGLE_API_KEY" not in os.environ:
    try:
        from google.colab import userdata
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except ImportError:
        pass
        
if "OPENAI_API_KEY" not in os.environ:
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except ImportError:
        pass

## 1. Direct SDKs Comparison

### Google GenAI Client

In [ ]:
from google import genai
from google.genai import types

sysmsg = "Imagine you're a travel planner, answer carefully"

# Google GenAI SDK (using Gemini 2.5-flash)
client = genai.Client()
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Write one line on AI.",
    config=types.GenerateContentConfig(system_instruction=sysmsg)
)
print(response.text)

### OpenAI Client

In [ ]:
from openai import OpenAI

# OpenAI SDK (using gpt-4o-mini)
client_openai = OpenAI()
completion = client_openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": sysmsg},
        {"role": "user", "content": "Write one line on AI."}
    ],
)
print(completion.choices[0].message.content)

## 2. Using LangChain as an Orchestrator

LangChain abstracts these SDKs into a unified interface, allowing you to swap backends easily without altering the rest of your pipeline.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Using Gemini via LangChain
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")
response = llm.invoke("Explain LangChain in simple words.")
print(response.content)

### Structured Message Inputs

LangChain models receive lists of message objects (`SystemMessage`, `HumanMessage`, `AIMessage`).

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

system_prompt = SystemMessage(content="Imagine you're a travel planner, answer carefully in one line.")
user_message = HumanMessage(content="Tell me about Paris.")

response = llm.invoke([system_prompt, user_message])
print(response.content)

## 3. Stateful Conversations: Manual Loop

By passing the conversational history list along with each new invocation, we can maintain chat history manually.

In [ ]:
from langchain_core.messages import AIMessage

chat_history = []
system_prompt = SystemMessage(content="Imagine you're a travel planner, answer in one line.")

print("Type 'exit' to stop the loop.\n")
while True:
    user_query = input("User: ")
    if user_query.lower() == "exit":
        break
        
    user_message = HumanMessage(content=user_query)
    
    # Pack system prompt, history, and the new message
    full_payload = [system_prompt] + chat_history + [user_message]
    response = llm.invoke(full_payload)
    
    # Append messages to update state
    chat_history.append(user_message)
    chat_history.append(AIMessage(content=response.content))
    
    print(f"AI: {response.content}\n")

## 4. Behind the Scenes: Debugging & Tracing

LangChain provides global variables to inspect the underlying prompts and inputs. Enabling `set_debug(True)` outputs the detailed chain execution flow, showing you exactly what is sent to the LLM backend.

In [ ]:
import langchain

# Turn on global debugging to see raw prompts and execution traces
langchain.globals.set_debug(True)

print("--- Debug Run ---")
response = llm.invoke("Say 'Debugging is active'")

# Remember to turn it off afterwards to keep the console clean
langchain.globals.set_debug(False)
print("--- Debug Disabled ---")

## 5. Model Parameters & Configurations

You can fine-tune LLM responses by passing configurations such as `temperature` (randomness control), `max_tokens`, and parameters specifically bound to the model during initialization.

In [ ]:
# Instantiate the model with customized parameters
strict_model = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0.1,  # Lower temperature results in more factual, reproducible answers
    max_tokens=50     # Limit output length
)

response = strict_model.invoke("List three primary colors in bullet points.")
print(response.content)